In [1]:
# Competition-Solution/notebooks/trial/00_baseline_model_training.ipynb

### Baseline Model Training

The notebooks within the **'02_model_training'** folder perform the training of a baseline model based on the trial dataset, focusing on the three subtasks:

- **Subtask 1:** Call2Action

- **Subtask 2:** Attacks on the Democratic Basic Order (FDGO)

- **Subtask 3:** Violence Detection


The goal of this baseline model training is to implement and validate a fine-tuning pipeline using a pre-trained Transformer model.

The insights gained from this initial baseline training will serve these purposes:
*   Provide a reference point against which all subsequent approaches will be compared to.
*   Help identify early on if there are any subtask-specific challenges that might require tailored strategies to resolve later.
*   Validate the overall training and evaluation pipeline before moving to the larger training dataset.

### Codabench Baseline Metrics

The Codabench contest provided baseline results to offer a starting benchmark. Their baselines were formed based on the full training dataset. The approaches and their reported Macro-F1 scores for each subtask are as follows:

*   **Subtask 1: Call2Action (Binary Classification)**
    *   **Baseline Approach:** Gradient-boosting classifier with SentenceBert embeddings and tweet polarity, using undersampling.
    *   **Reported Macro-F1 Score:** 0.59

*   **Subtask 2: Attacks on the Democratic Basic Order (FDGO) (Multi-class Classification)**
    *   **Baseline Approach:** Linear Support Vector Machine (SVM) with TF-IDF weighted bag-of-phrases (unigrams and bigrams), using a cost-sensitive SVM.
    *   **Reported Macro-F1 Score:** 0.47

*   **Subtask 3: Violence Detection (Binary Classification)**
    *   **Baseline Approach:** Large Language Model Qwen2.5 (32 billion parameters) in a few-shot scenario.
    *   **Reported Macro-F1 Score:** 0.69

These scores serve as an initial comparison point for the subsequent approaches and training runs.

---

##### <b>Imports</b>

In [ ]:
import sys
from pathlib import Path
from rich.console import Console
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

import wandb
import torch

# Hugging Face Imports
from transformers import (
  AutoProcessor,
  AutoModelForSequenceClassification,
  DataCollatorWithPadding,
  Trainer,
  TrainingArguments,
  EarlyStoppingCallback
)
from datasets import load_from_disk
import evaluate

console = Console()

# Path to this notebook
notebook_dir = Path.cwd()

# Project root directory
project_root_dir = notebook_dir.parent.parent.parent

# Source path
src_path = project_root_dir / "src"
sys.path.append(str(src_path))

console.print(f"Project root: {project_root_dir}", style="cyan")
console.print(f"Source path: {src_path}", style="cyan")
console.print(f"Source path exists: {src_path.exists()}", style="cyan")

# Local imports
import wandb_utils
import config_utils

### **Experiment Configuration**

In [ ]:
# Default values for configuration choices
default_config_choices = {
    "CHOSEN_SUBTASK_DIR_NAME": "subtask1_call2action",
    "CHOSEN_EXPERIMENT_FILE_NAME": "EuroBERT_baseline.yaml",
    "CHOSEN_DATASET_MODE": "trial",
    "RUN_VERSION_TAG": "v1",
    "LOG_TO_WANDB": True
}

# Initializes default config choices
global_config_choices = default_config_choices.copy()

console.print("Initial/Default Configuration Choices:", style="bold yellow")
for config_option, config_choice in global_config_choices.items():
    console.print(f"  {config_option}: {config_choice}")

In [ ]:
# Option values
subtask_options = [
    ("Sub-task 1 · Call-to-Action", "subtask1_call2action"),
    ("Sub-task 2 · Attacks on DBO", "subtask2_attacks_on_dbo"),
    ("Sub-task 3 · Violence", "subtask3_violence"),
]

experiment_options = [
    ("Euro-BERT · baseline", "EuroBERT_baseline.yaml"),
    ("Modern-GBERT · baseline", "ModernGBERT_baseline.yaml"),
    ("Modern-GBERT · train", "ModernGBERT_train.yaml"),
]

dataset_mode_options = [
    ("Trial", "trial"),
    ("Train", "train"),
]

# Form Widgets
# Subtask Dropdown
dd_subtask = widgets.Dropdown(
    options=subtask_options,
    value=global_config_choices['CHOSEN_SUBTASK_DIR_NAME'],
    description="Sub-task:",
    layout=widgets.Layout(width="320px"),
    style={"description_width": "90px"}
)

# Experiment Dropdown
dd_experiment = widgets.Dropdown(
    options=experiment_options,
    value=global_config_choices['CHOSEN_EXPERIMENT_FILE_NAME'],
    description="Experiment:",
    layout=widgets.Layout(width="320px"),
    style={"description_width": "90px"}
)

# Dataset Mode Dropdown
dd_mode = widgets.Dropdown(
    options=dataset_mode_options,
    value=global_config_choices['CHOSEN_DATASET_MODE'],
    description="Dataset mode:",
    layout=widgets.Layout(width="320px"),
    style={"description_width": "90px"}
)

# Run Version Text Input
txt_version = widgets.Text(
    value=global_config_choices['RUN_VERSION_TAG'],
    description="Run tag:",
    placeholder="e.g. v1",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="200px")
)

# Log to W&B Checkbox
log_to_wandb = widgets.Checkbox(
    description="Log to W&B",
    value=global_config_choices['LOG_TO_WANDB'],
    layout=widgets.Layout(width="200px"),
    style={"description_width": "90px"}
)

# Save Button
save_btn = widgets.Button(
    description="Save choices",
    button_style="success",
    icon="check"
)

status_out = widgets.Output()

def _on_run_clicked(b):
    global global_config_choices
    global_config_choices = {
        "CHOSEN_SUBTASK_DIR_NAME": dd_subtask.value,
        "CHOSEN_EXPERIMENT_FILE_NAME": dd_experiment.value,
        "CHOSEN_DATASET_MODE": dd_mode.value,
        "RUN_VERSION_TAG": txt_version.value,
        "LOG_TO_WANDB": log_to_wandb.value
    }
    with status_out:
        clear_output(wait=True)
        console.print("Config saved:", global_config_choices, style="green")

save_btn.on_click(_on_run_clicked)

# Builds the form
form = widgets.VBox([
    widgets.HTML("<h4 style='margin:0 0 8px 0'>Configure experiment</h4>"),
    dd_subtask,
    dd_experiment,
    dd_mode,
    txt_version,
    log_to_wandb,
    save_btn,
    status_out
])

display(form)

In [ ]:
# Path to the configs directory
config_base_dir_nb = project_root_dir / "configs"

# Defines the paths to the global, subtask and experiment configs
base_cfg_path = config_base_dir_nb / "base.yaml" # Global base config
subtask_base_cfg_path = config_base_dir_nb / global_config_choices['CHOSEN_SUBTASK_DIR_NAME'] / "base.yaml" # Subtask base config
experiment_cfg_path = config_base_dir_nb / global_config_choices['CHOSEN_SUBTASK_DIR_NAME'] / global_config_choices['CHOSEN_EXPERIMENT_FILE_NAME'] # Experiment config

# Loads the global, subtask and experiment configs
cfg = config_utils.load_config(
  base_config_path=base_cfg_path,
  subtask_config_path=subtask_base_cfg_path,
  experiment_config_path=experiment_cfg_path
)

# Raises an error if one of the config files is not found
if not cfg:
  raise ValueError(f'Configuration files could not be loaded. Please check the config YAML files in the \"configs\" directory and widget selections:\n'
                   f"    Global base config: {base_cfg_path}\n"
                   f"    Subtask base config: {subtask_base_cfg_path}\n"
                   f"    Experiment config: {experiment_cfg_path}")

console.print(f"Successfully loaded and merged configurations for: \n"
              f"{global_config_choices['CHOSEN_SUBTASK_DIR_NAME']} / "
              f"{global_config_choices['CHOSEN_EXPERIMENT_FILE_NAME']} "
              f"(Mode: {global_config_choices['CHOSEN_DATASET_MODE']})", style="green")

##### <b>Loading the HF datasets</b>

In [ ]:
# Defines the path to the processed data directory
processed_data_root_dir = project_root_dir / cfg['paths']['processed_data_dir_name']
console.print(f"Processed Data Root Directory: '{processed_data_root_dir}'", style="bold")

# Defines the suffix of the dataset (e.g. dbo/dbo_hf_dataset)
dataset_load_path_suffix = cfg['dataset_details']['hf_dataset_path_suffix']
console.print(f"Dataset Load Path Suffix: '{dataset_load_path_suffix}'", style="bold")

# Constructs the full path to the dataset by joining the processed data root directory and the dataset load path suffix
full_dataset_load_path = processed_data_root_dir / dataset_load_path_suffix

console.print(f"Loading dataset for subtask '{cfg['subtask_name']}' from: '{full_dataset_load_path}'", style="green")

# Tries to load the dataset from the full path
try:
    raw_dataset = load_from_disk(str(full_dataset_load_path))
    console.print(f"Successfully loaded dataset: \n{raw_dataset}", style="green")
except FileNotFoundError:
    console.print(f"ERROR: Dataset not found at {full_dataset_load_path}. Please check your config and data.", style="bold red")
    raise
except Exception as e:
    console.print(f"ERROR: Could not load dataset from {full_dataset_load_path}: {e}", style="bold red")
    raise

In [ ]:
# Model configuration and paths
console.print(f"Model: {cfg['model_checkpoint']}", style="bold")
console.print(f"Model config: {cfg['model_config']}", style="bold")

# Checks for local model first, fallback to HF Hub
local_model_path = project_root_dir / cfg['paths']['base_models_dir_name'] / cfg['model_checkpoint']
if local_model_path.exists():
    model_path = str(local_model_path)
    console.print(f"Found local model at: {model_path}", style="cyan")
else:
    model_path = cfg['model_checkpoint']
    console.print(f"Local model not found, using Hugging Face Hub: {model_path}", style="yellow")

# Loads the model
try:
    model = AutoModelForSequenceClassification.from_pretrained(
        model_path,
        **cfg['model_config']
    )
    console.print(f"Successfully loaded model: {model.__class__.__name__}", style="green")
except Exception as e:
    console.print(f"ERROR: Could not load model from {model_path}: {e}", style="bold red")
    raise

# If a gpu is available, moves the model to the gpu
if torch.cuda.is_available():
    model.to("cuda")
    console.print("Model moved to GPU", style="green")

In [ ]:
console.print(f"Loading AutoProcessor for '{model_path}'", style="green")

# Loads the processor and adding the special tokens (Anonymization tokens that we set during the data preprocessing phase)
processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
processor.add_special_tokens({"additional_special_tokens": cfg['tokenization']['special_tokens']})
model.resize_token_embeddings(len(processor), mean_resizing=True)
console.print(f"Processor loaded with {len(cfg['tokenization']['special_tokens'])} special tokens, total: {len(processor)}", style="cyan")

Total number of unique tokens: **128262**

Dimensionality of the token embeddings: **768**

ID of padding token: **128001**

In [ ]:
console.print(f"Tokenizing dataset for subtask '{cfg['subtask_name']}'...", style="green")

# Tokenization function
def tokenize_function(examples):
    return processor(
        examples[cfg['tokenization']['text_column_name']],
        padding=cfg['tokenization']['padding'],
        truncation=cfg['tokenization']['truncation'],
    )
# Applies the tokenization mapping function to the raw dataset
try:
    tokenized_dataset = raw_dataset.map(tokenize_function, batched=True)
    console.print(f"Dataset tokenized successfully: {tokenized_dataset}", style="bold green")
except Exception as e:
    console.print(f"ERROR during dataset tokenization: {e}", style="bold red")
    raise

# Renames the subtasks respective label column to "labels", as the Trainer expects it
tokenized_dataset = tokenized_dataset.rename_column(cfg['dataset_details']['label_column'], "labels")
console.print(f"Renamed '{cfg['dataset_details']['label_column']}' to \"labels\"", style="green")

# Saves the tokenized dataset to disk
save_path = processed_data_root_dir / cfg['dataset_details']['hf_dataset_tokenized_path_suffix']
tokenized_dataset.save_to_disk(str(save_path))
console.print(f"Tokenized dataset saved to: {save_path}", style="green")

In [ ]:
# Defines the data collator that is used to collate the tokenized dataset (per batch)
# (Pads the tokenized inputs to the same length, Stacks the tokenized inputs into a batch, Returns the tokenized inputs as PyTorch tensors)
data_collator = DataCollatorWithPadding(
    tokenizer=processor,
    return_tensors="pt"
)
console.print("Data collator configured for PyTorch tensors", style="cyan")

- [Full List of Training Arguments](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.TrainingArguments)

In [ ]:
# Builds the run name and directories
dataset_mode = global_config_choices['CHOSEN_DATASET_MODE'] # e.g. 'trial' or 'train'
run_version = global_config_choices['RUN_VERSION_TAG'] # e.g. 'v1'

# Builds the run name e.g. 'trial-c2a-eurobert210m-baseline-v1'
run_name = f"{cfg['dataset_modes'][dataset_mode]['suffix']}-{cfg['subtask_id']}-{cfg['wandb_run_name_parts']['model_arch']}-{cfg['experiment_type']}"
if run_version:
    run_name += f"-{run_version}"

# Builds the output and logging directory paths
output_dir = project_root_dir / cfg['paths']['output_dir_base'] / cfg['subtask_id'] / run_name
logging_dir = project_root_dir / cfg['paths']['logging_dir_base'] / dataset_mode / run_name

console.print(f"Training run: {run_name}", style="bold yellow")
console.print(f"Early stopping patience: {cfg['callbacks']['early_stopping_patience']}", style="cyan")

# Creates the training arguments (Uses the configs settings)
training_args = TrainingArguments(
    output_dir=str(output_dir),
    logging_dir=str(logging_dir),
    run_name=run_name,
    **cfg['training_arguments']
)

console.print(f"Training arguments: {training_args}", style="cyan")

In [ ]:
# Loads the evaluation metrics from the config
loaded_metrics = {}

for metric in cfg['evaluation']['metrics_to_load']:
    loaded_metrics[metric] = evaluate.load(metric)
for metric in loaded_metrics:
    console.print(f"Loaded metric: {metric}", style="green")

# Defines the compute_metrics function that is used to compute the evaluation metrics during training
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    
    return {
        "f1-macro": loaded_metrics['f1'].compute(predictions=preds, references=labels, average=cfg['evaluation']['f1_average_type'])['f1'],
        "accuracy": loaded_metrics['accuracy'].compute(predictions=preds, references=labels)['accuracy'],
        "precision-macro": loaded_metrics['precision'].compute(predictions=preds, references=labels, average=cfg['evaluation']['precision_average_type'])['precision'],
        "recall-macro": loaded_metrics['recall'].compute(predictions=preds, references=labels, average=cfg['evaluation']['recall_average_type'])['recall']
    }

In [ ]:
# Clears the GPU cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Initializes the Trainer
trainer = Trainer(
    model=model,
    processing_class=processor,
    data_collator=data_collator,
    args=training_args,
    train_dataset=tokenized_dataset[cfg['dataset_splits']['train']],
    eval_dataset=tokenized_dataset[cfg['dataset_splits']['validation']],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=cfg['callbacks']['early_stopping_patience'])]
)

console.print(f"Trainer initialized with early stopping patience: {cfg['callbacks']['early_stopping_patience']}", style="green")
console.print(f"Training on {len(tokenized_dataset[cfg['dataset_splits']['train']])} samples", style="cyan")
console.print(f"Validating on {len(tokenized_dataset[cfg['dataset_splits']['validation']])} samples", style="cyan")

In [ ]:
# Starts the Tensorboard Dashboard in the Jupyter Notebook (Requires the Tensorboard IDE extension)
# Otherwise, the Tensorboard Dashboard is also available at http://localhost:6006/
%load_ext tensorboard
%tensorboard --logdir {str(logging_dir)}

In [ ]:
# Finishes any existing W&B run
if wandb.run is not None:
    console.print("Finishing previous W&B run...", style="yellow")
    wandb.finish()

# Initializes W&B if we chose to log to W&B in the jupyter widget at the beginning of the notebook
if global_config_choices['LOG_TO_WANDB']:
    console.print("W&B logging enabled. Attempting login...", style="yellow")
    
    # Logs in to W&B and initializes run if successful
    if wandb_utils.login_wandb(cfg['wandb']['dotenv_path']):
        console.print("W&B login successful.", style="bold green")
        
        try:
            wandb_run = wandb_utils.init_wandb(
                loaded_config=cfg,
                dataset_mode=global_config_choices['CHOSEN_DATASET_MODE'],
                logging_dir=project_root_dir / cfg['paths']['logging_dir_base'],
                version_tag=global_config_choices['RUN_VERSION_TAG']
            )
            
            # Adds W&B to training args and sets up model watching
            if "wandb" not in training_args.report_to:
                training_args.report_to.append("wandb")
                console.print("W&B added to training args", style="green")
            
            wandb_run.watch(
                models=model,
                log=cfg['wandb']['watch_model_log'], 
                log_freq=cfg['wandb']['watch_model_log_freq']
            )
            console.print(f"W&B model watching enabled: {cfg['wandb']['watch_model_log']} (freq: {cfg['wandb']['watch_model_log_freq']})", style="cyan")
            console.print(f"W&B run initialized: {wandb_run.name} (ID: {wandb_run.id})", style="bold green")
            
        except Exception as e:
            console.print(f"ERROR: Failed to initialize W&B run: {e}", style="bold red")
            wandb_run = None
    else:
        console.print("W&B login failed. Training will proceed without W&B logging.", style="bold red")
        wandb_run = None
else:
    console.print("W&B logging disabled in the widget.", style="yellow")
    wandb_run = None

In [ ]:
# Creates a jupyter widget button to start the training loop
train_btn = widgets.Button(
    description="Train model",
    icon="play",
    button_style="success",
    tooltip="Start the training loop",
    layout=widgets.Layout(width="160px")
)
# Creates a jupyter widget output to display the training logs
log_out = widgets.Output(
    layout=widgets.Layout(border="1px solid #ccc",
                    max_height="350px",
                    overflow="auto",
                    padding="4px")
)
# Defines the function that is called when the training button is clicked
def _train_model(btn):
    # Disables the training button after starting the training run
    train_btn.disabled = True
    # Clears the log output when starting a new training run
    with log_out:
        clear_output(wait=True)
        console.print("Training started...", style="bold green")

    # Tries to launch the training run
    try:
        training_result = trainer.train()
        with log_out:
            console.print("Training finished successfully!", style="bold green")
            if training_result.metrics:
                console.print("Final metrics:", style="cyan")
                for metric, metric_value in training_result.metrics.items():
                    console.print(f"  {metric}: {metric_value:.4f}", style="white")
    except Exception as e:
        with log_out:
            console.print(f"Training failed with error: {e}", style="bold red")
            raise
    finally:
        # Resets the training button to allow for another training run
        train_btn.disabled = False

# Adds the training button to the form
train_btn.on_click(_train_model)

# Displays the training button and log output
display(widgets.VBox([
    widgets.HTML("<h4 style='margin:0 0 8px 0'>Run experiment</h4>"),
    train_btn,
    log_out
]))


In [ ]:
# Saves the best model (Do not run this cell before the "Train model" widget has finished, as it will interrupt the trianing)
trainer.save_model(f"{str(output_dir)}/best_model_{cfg['training_arguments']['metric_for_best_model']}")

In [ ]:
# Saves the model and creates a W&B artifact
if wandb_run is not None:
    try:
        console.print("Saving model and creating W&B artifact...", style="yellow")
        
        # Builds the artifact name, description, and target path
        artifact_name = cfg['wandb']['artifact']['name_template'].format(
            model_arch=cfg['wandb_run_name_parts']['model_arch'],
            subtask_id=cfg['subtask_id'],
            experiment_type=cfg['experiment_type']
        ) # e.g. "eurobert210m_c2a_baseline"
        
        description = cfg['wandb']['artifact']['description_template'].format(
            model_arch=cfg['wandb_run_name_parts']['model_arch'],
            subtask_name=cfg['subtask_name'],
            experiment_type=cfg['experiment_type'],
            dataset_mode_suffix=cfg['dataset_modes'][global_config_choices['CHOSEN_DATASET_MODE']]['suffix']
        ) # e.g. "Fine-tuned EuroBERT for Call2Action (GermEval 2025)"
        
        # Ensures the target path is in "collection/alias" or "project/collection/alias" format for linking
        target_path = cfg['wandb']['artifact']['portfolio_path_template'].format(
            project=cfg['wandb']['project'],
            model_arch=cfg['wandb_run_name_parts']['model_arch'],
            subtask_id=cfg['subtask_id']
        ) # e.g. Bachelors-Thesis/eurobert210m_c2a_models"
        
        console.print(f"Creating artifact: {artifact_name}", style="cyan")
        console.print(f"Description: {description}", style="cyan")
        console.print(f"Local model path: {output_dir}", style="cyan")
        console.print(f"Target W&B path for linking: {target_path}", style="cyan")
        
        artifact = wandb_utils.save_and_upload_model_to_wandb(
            run=wandb_run,
            name=artifact_name,
            model_type=cfg['wandb']['artifact']['model_type'],
            description=description,
            metadata={
                'model_checkpoint': cfg['model_checkpoint'],
                'subtask_id': cfg['subtask_id'],
                'subtask_name': cfg['subtask_name'],
                'experiment_type': cfg['experiment_type'],
                'dataset_mode': global_config_choices['CHOSEN_DATASET_MODE'],
                'run_name': wandb_run.name,
                'training_args': cfg['training_arguments'],
                'final_metrics': trainer.state.log_history[-1] if trainer.state.log_history else {}
            },
            local_path=str(output_dir), # Path to the local directory containing the model files
            target_path=target_path # Path in W&B to link this artifact version
        )
        
    except Exception as e:
        console.print(f"ERROR: Failed to save model artifact to W&B: {e}", style="bold red")
        console.print("Model was saved locally but W&B artifact creation/linking failed.", style="yellow")
    
    finally:
        console.print("Finishing W&B run...", style="yellow")
        wandb_run.finish()
        console.print("W&B run finished.", style="green")
else:
    console.print("No W&B run to finish (W&B was disabled or failed to initialize).", style="yellow")

console.print(f"Training completed! Model saved to: {target_path}", style="bold green")